# 🧠 GAN FIFA Panini — Parte 1: Entrenamiento DCGAN
## Centro de Competencias Digitales · UNAB

> **Objetivo:** Entrenar una DCGAN con imágenes de futbolistas FIFA y guardar el generador entrenado en formato `.keras` para usarlo en la app Streamlit.

### 🗺️ Ruta

| Paso | Descripción |
|------|-------------|
| **1** | Instalación y configuración |
| **2** | Descarga y preparación del dataset FIFA |
| **3** | Aumentación de datos |
| **4** | Construcción de la DCGAN (Generador + Discriminador) |
| **5** | Entrenamiento con monitoreo visual |
| **6** | 💾 Guardar el generador → `generador_fifa_ccd.keras` |

> ⚙️ **Importante:** Ejecuta este notebook en **Google Colab con GPU activada**  
> `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`

> 📦 **Salida:** Al final descargarás `generador_fifa_ccd.keras` para subirlo a Streamlit Cloud.

## 📦 Paso 1: Instalación de librerías y configuración

In [ ]:
# ── Instalación de dependencias ───────────────────────────────────────────────
!pip install -q gdown pillow

import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Verificar GPU
print('=' * 50)
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU detectada: {gpus[0].name}')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('⚠️  No se detectó GPU — actívala en Entorno de ejecución → T4 GPU')
print('=' * 50)

## 📥 Paso 2: Descarga y preparación del dataset FIFA

Descargamos el conjunto de imágenes de rostros de futbolistas y eliminamos fondos con canal alfa.

In [ ]:
# ── Descarga del dataset ──────────────────────────────────────────────────────
!gdown https://drive.google.com/uc?id=1zpT6gHzvbr21_Vq4NjpRY0JGSaQOSANa -O fifa.zip
!unzip -q fifa.zip -d .
print('✅ Dataset descargado y extraído')

In [ ]:
# ── Funciones de preprocesamiento ─────────────────────────────────────────────

def remove_noise(image):
    """Elimina el fondo transparente del canal alfa de imágenes PNG."""
    alpha = image[:, :, 3] > 50
    alpha = tf.cast(alpha, tf.uint8)
    alpha = tf.expand_dims(alpha, axis=-1)
    noise_filtered = tf.multiply(image, alpha)
    return noise_filtered[:, :, :3]

def read_voc_images(voc_dir, n=-1):
    """Carga imágenes de una carpeta, eliminando fondos transparentes."""
    files = [os.path.join(voc_dir, f) for f in os.listdir(voc_dir)]
    files = sorted(files, key=lambda i: int(os.path.splitext(os.path.basename(i))[0]))

    features, labels = [], []
    errores = 0
    for i, fname in enumerate(files):
        if n != -1 and i == n:
            break
        try:
            raw = tf.io.read_file(fname)
            img = tf.io.decode_image(raw, channels=4, expand_animations=False)
            img = remove_noise(img)
            features.append(img)
            label_id = int(os.path.splitext(os.path.basename(fname))[0])
            labels.append(label_id)
        except Exception:
            errores += 1
            continue

    print(f'✅ Imágenes cargadas: {len(features)} | ⚠️ Errores ignorados: {errores}')
    return features, labels

# Cargar CSV y dataset
data = pd.read_csv('data.csv')
caracteristicas, l = read_voc_images('/content/Images')
datos = data.loc[l]
nombres = list(datos['Name'])
print(f'📊 Total de jugadores cargados: {len(nombres)}')

In [ ]:
# ── Visualizar muestra del dataset ────────────────────────────────────────────

def mostrar_grid(imagenes, titulos=None, filas=2, cols=5, escala=2.2):
    fig, axes = plt.subplots(filas, cols, figsize=(cols * escala, filas * escala))
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        if i >= len(imagenes):
            ax.axis('off')
            continue
        img = imagenes[i].numpy() if tf.is_tensor(imagenes[i]) else imagenes[i]
        ax.imshow(img.astype(np.uint8))
        ax.axis('off')
        if titulos and i < len(titulos):
            ax.set_title(titulos[i], fontsize=7, pad=2)
    plt.suptitle('🏟️ Muestra del dataset FIFA', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

mostrar_grid(caracteristicas[:10], titulos=nombres[:10])

## 🔄 Paso 3: Aumentación de datos

La aumentación de datos es una técnica clave para mejorar la generalización de la GAN.

| Técnica | Descripción | Beneficio |
|---------|-------------|-----------|
| **Flip horizontal** | Espeja la imagen | Duplica la variedad visual |
| **Rotación aleatoria** | Rota ±10° | Robustez ante orientación |
| **Ajuste de brillo** | ±20% de brillo | Variedad de iluminación |
| **Ajuste de contraste** | ±20% | Diversidad visual |
| **Recorte y zoom** | Crop aleatorio | Invariancia a escala |

In [ ]:
# ── Pipeline de aumentación con tf.data ───────────────────────────────────────
IMG_SIZE = 64
Z_DIM    = 100
BATCH    = 64

@tf.function
def aumentar_imagen(img):
    """Aplica aumentación aleatoria a una imagen normalizada [-1, 1]."""
    img_01 = (img + 1.0) / 2.0
    img_01 = tf.image.random_flip_left_right(img_01)
    img_01 = tf.image.random_brightness(img_01, max_delta=0.2)
    img_01 = tf.image.random_contrast(img_01, lower=0.8, upper=1.2)
    img_01 = tf.clip_by_value(img_01, 0.0, 1.0)
    crop_size = tf.random.uniform([], minval=52, maxval=64, dtype=tf.int32)
    img_01 = tf.image.random_crop(img_01, size=[crop_size, crop_size, 3])
    img_01 = tf.image.resize(img_01, [IMG_SIZE, IMG_SIZE])
    return img_01 * 2.0 - 1.0

def preprocesar_imagen(img):
    """Redimensiona y normaliza a [-1, 1] sin aumentación."""
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return (img / 127.5) - 1.0

def crear_dataset(imagenes, etiquetas, nombres, batch_size=BATCH, aumentar=True):
    """Crea pipeline tf.data con preprocesamiento y aumentación opcional."""
    imgs_proc = [preprocesar_imagen(img) for img in imagenes]
    etiquetas_t = tf.convert_to_tensor(etiquetas, dtype=tf.int32)
    nombres_t   = tf.convert_to_tensor(nombres, dtype=tf.string)
    dataset = tf.data.Dataset.from_tensor_slices((imgs_proc, etiquetas_t, nombres_t))
    dataset = dataset.shuffle(buffer_size=len(imagenes))
    if aumentar:
        dataset = dataset.map(
            lambda img, et, nom: (aumentar_imagen(img), et, nom),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

fifa_dataset = crear_dataset(caracteristicas, l, nombres, aumentar=True)
print(f'✅ Dataset listo con aumentación | Batches: {len(list(fifa_dataset))}')

In [ ]:
# ── Visualizar aumentación aplicada ───────────────────────────────────────────
img_original = preprocesar_imagen(caracteristicas[0])
imgs_aumentadas = [aumentar_imagen(img_original) for _ in range(8)]

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
axes = axes.flatten()
for i, ax in enumerate(axes):
    img_vis = ((imgs_aumentadas[i].numpy() + 1.0) / 2.0 * 255).astype(np.uint8)
    ax.imshow(img_vis)
    ax.set_title(f'Aumentada #{i+1}', fontsize=9)
    ax.axis('off')
plt.suptitle('🔄 Ejemplos de aumentación de datos', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 🧠 Paso 4: Arquitectura DCGAN mejorada

### ¿Cómo funciona una GAN?

```
Ruido aleatorio (z) → [GENERADOR] → Imagen falsa
                                          ↓
Imagen real → [DISCRIMINADOR] → ¿Real o Falsa?
```

**El Generador** aprende a engañar al Discriminador.  
**El Discriminador** aprende a no ser engañado.  
Con el tiempo, las imágenes generadas se vuelven cada vez más realistas.

### Mejoras aplicadas:
- ✅ Dropout en el Discriminador para regularización
- ✅ Label smoothing en pérdidas
- ✅ Gaussian noise en entradas del Discriminador

In [ ]:
# ── Bloques de construcción ───────────────────────────────────────────────────
from tensorflow.keras import layers

def G_block(out_channels, kernel_size=4, strides=2, padding='same'):
    """Bloque del Generador: ConvTranspose + BatchNorm + ReLU."""
    return tf.keras.Sequential([
        layers.Conv2DTranspose(out_channels, kernel_size, strides=strides,
                               padding=padding, use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU()
    ])

def construir_generador(n_G=64, z_dim=Z_DIM):
    """
    Generador DCGAN: transforma ruido z → imagen 64x64 RGB.
    Arquitectura: z(100) → 4x4x512 → 8x8 → 16x16 → 32x32 → 64x64x3
    """
    modelo = tf.keras.Sequential([
        layers.Input(shape=(z_dim,)),
        layers.Dense(4 * 4 * n_G * 8, use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Reshape((4, 4, n_G * 8)),
        G_block(n_G * 4),
        G_block(n_G * 2),
        G_block(n_G),
        layers.Conv2DTranspose(3, kernel_size=4, strides=2,
                               padding='same', use_bias=False),
        layers.Activation('tanh')
    ], name='Generador')
    return modelo

def construir_discriminador(n_D=64, dropout_rate=0.3):
    """
    Discriminador DCGAN mejorado con Dropout.
    Arquitectura: 64x64x3 → conv layers → logit escalar
    """
    modelo = tf.keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.GaussianNoise(0.05),
        layers.Conv2D(n_D, 4, strides=2, padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(dropout_rate),
        layers.Conv2D(n_D * 2, 4, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Dropout(dropout_rate),
        layers.Conv2D(n_D * 4, 4, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Dropout(dropout_rate),
        layers.Conv2D(n_D * 8, 4, strides=2, padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Flatten(),
        layers.Dense(1)
    ], name='Discriminador')
    return modelo

# Instanciar modelos
generador     = construir_generador()
discriminador = construir_discriminador()

generador.summary()
print()
discriminador.summary()

## ⚙️ Paso 5: Funciones de pérdida y entrenamiento

### Label Smoothing
En lugar de usar etiquetas duras (0 y 1), usamos valores suavizados:
- Imágenes reales → **0.9** (en vez de 1.0)
- Imágenes falsas → **0.1** (en vez de 0.0)

Esto evita que el discriminador se vuelva demasiado seguro y mejora la estabilidad.

In [ ]:
# ── Pérdidas con Label Smoothing ──────────────────────────────────────────────
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

SMOOTH_REAL = 0.9
SMOOTH_FAKE = 0.1

def loss_discriminador(real_logits, fake_logits):
    """Pérdida del discriminador con label smoothing."""
    real_loss = cross_entropy(
        tf.ones_like(real_logits) * SMOOTH_REAL, real_logits)
    fake_loss = cross_entropy(
        tf.ones_like(fake_logits) * SMOOTH_FAKE, fake_logits)
    return real_loss + fake_loss

def loss_generador(fake_logits):
    """El generador quiere que el discriminador crea que sus imágenes son reales."""
    return cross_entropy(tf.ones_like(fake_logits), fake_logits)

gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)

# Forzar construcción de variables
generador(tf.random.normal([1, Z_DIM]))
discriminador(tf.random.normal([1, IMG_SIZE, IMG_SIZE, 3]))
print('✅ Modelos compilados y listos para entrenar')

In [ ]:
# ── Paso de entrenamiento con @tf.function ────────────────────────────────────

@tf.function
def train_step(imagenes_reales):
    batch_size = tf.shape(imagenes_reales)[0]
    ruido = tf.random.normal([batch_size, Z_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        imgs_falsas   = generador(ruido, training=True)
        logits_reales = discriminador(imagenes_reales, training=True)
        logits_falsas = discriminador(imgs_falsas, training=True)
        loss_D = loss_discriminador(logits_reales, logits_falsas)
        loss_G = loss_generador(logits_falsas)

    grad_G = gen_tape.gradient(loss_G, generador.trainable_variables)
    grad_D = disc_tape.gradient(loss_D, discriminador.trainable_variables)

    gen_optimizer.apply_gradients(zip(grad_G, generador.trainable_variables))
    disc_optimizer.apply_gradients(zip(grad_D, discriminador.trainable_variables))
    return loss_G, loss_D

# Ruido fijo para monitorear progreso con las mismas semillas
ruido_fijo = tf.random.normal([16, Z_DIM])

def mostrar_progreso(generador, ruido, epoch=None):
    """Muestra cuadrícula 4x4 de imágenes generadas."""
    imgs = generador(ruido, training=False)
    imgs = ((imgs + 1.0) / 2.0).numpy()

    fig, axes = plt.subplots(4, 4, figsize=(6, 6))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(np.clip(imgs[i], 0, 1))
        ax.axis('off')

    titulo = f'🎨 Imágenes generadas — Época {epoch}' if epoch else '🎨 Imágenes generadas'
    plt.suptitle(titulo, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Bucle de entrenamiento principal ──────────────────────────────────────────

# 🔧 AJUSTA AQUÍ: más épocas = mejor calidad (pero más tiempo)
EPOCHS     = 100    # Recomendado: 100-200 para resultados decentes
SHOW_EVERY = 10     # Mostrar imágenes cada N épocas

# Dataset solo imágenes (sin metadatos)
dataset_imgs = fifa_dataset.map(
    lambda img, et, nom: img, num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

historial_G, historial_D = [], []

print('🚀 Iniciando entrenamiento...')
print('─' * 50)

for epoch in range(1, EPOCHS + 1):
    losses_G, losses_D = [], []

    for batch in dataset_imgs:
        lg, ld = train_step(batch)
        losses_G.append(float(lg))
        losses_D.append(float(ld))

    avg_G = np.mean(losses_G)
    avg_D = np.mean(losses_D)
    historial_G.append(avg_G)
    historial_D.append(avg_D)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Época {epoch:3d}/{EPOCHS} | Loss G: {avg_G:.4f} | Loss D: {avg_D:.4f}')

    if epoch % SHOW_EVERY == 0:
        mostrar_progreso(generador, ruido_fijo, epoch=epoch)

print('─' * 50)
print('✅ Entrenamiento completado')

In [ ]:
# ── Curva de pérdidas ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(historial_G, label='Generador', color='#00BCD4', linewidth=2)
ax.plot(historial_D, label='Discriminador', color='#FF5722', linewidth=2)
ax.set_xlabel('Época', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('📈 Evolución de pérdidas durante el entrenamiento', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 💾 Paso 6: Guardar el generador entrenado

> Solo guardamos el **Generador** — es el único que necesita la app Streamlit.
> El Discriminador solo sirve durante el entrenamiento.

**Después de descargar `generador_fifa_ccd.keras`:**
1. Súbelo a un repositorio de GitHub o a Google Drive con enlace público
2. La app Streamlit lo descargará automáticamente al arrancar

In [ ]:
# ── Guardar solo el generador ─────────────────────────────────────────────────
generador.save('generador_fifa_ccd.keras')
print('✅ Generador guardado: generador_fifa_ccd.keras')
print(f'   Tamaño: {os.path.getsize("generador_fifa_ccd.keras") / 1e6:.1f} MB')

# Verificación rápida: generar 4 imágenes de prueba
print('\n🔍 Verificación — generando 4 imágenes de prueba con semillas fijas...')
tf.random.set_seed(42)
ruido_test = tf.random.normal([4, Z_DIM])
imgs_test = generador(ruido_test, training=False)
imgs_test = ((imgs_test + 1.0) / 2.0).numpy()

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i, ax in enumerate(axes):
    ax.imshow(np.clip(imgs_test[i], 0, 1))
    ax.set_title(f'Semilla {i+1}', fontsize=10)
    ax.axis('off')
plt.suptitle('✅ Verificación del generador guardado', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Descargar el modelo desde Colab ──────────────────────────────────────────
from google.colab import files
files.download('generador_fifa_ccd.keras')
print('📥 Descarga iniciada. Guarda este archivo — lo necesitas para la app Streamlit.')

## 🧩 Preguntas de reflexión

1. **¿Qué diferencias visuales observas** entre las imágenes generadas en la época 1 y la época final?
2. **¿Qué sucede** si cambias la tasa de aprendizaje a `1e-3`? ¿Y a `1e-6`?
3. **¿Por qué es importante** el Label Smoothing en el entrenamiento de GANs?
4. **¿Qué significa** que el Loss del Generador y el Discriminador oscilen sin converger a 0?
5. **Experimento avanzado:** ¿Qué cambiarías en la arquitectura para generar imágenes de **128x128** px?

---

> **Créditos:** Centro de Competencias Digitales (CCD) · UNAB · Bucaramanga, Colombia